In [0]:
# === J8 - Multi-day DistriNet pipeline setup ===
import sys
import os
from pathlib import Path

repo_root = "/Workspace/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline"
sys.path.insert(0, repo_root)

import importlib
for mod in ['src.ingestion', 'src.cleaning', 'src.transformation']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from src.ingestion import ingest_parquet_to_bronze_memory, clean_column_name
from src.cleaning import clean_bronze_to_silver
from src.transformation import transform_silver_to_gold

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName("j8-multi-day").getOrCreate()
spark.sql("USE CATALOG workspace")

print(f"✓ Spark version: {spark.version}")
print(f"✓ Catalog: workspace")

In [0]:
# === List all Parquet files in the raw_data volume ===
volume_path = "/Volumes/workspace/cybersecurity/raw_data"

import os
files = sorted(os.listdir(volume_path))
parquet_files = [f for f in files if f.endswith('.parquet')]

print(f"✓ Files in {volume_path}:")
for i, f in enumerate(parquet_files):
    full_path = os.path.join(volume_path, f)
    size_mb = os.path.getsize(full_path) / (1024 * 1024)
    print(f"  {i+1}. {f} ({size_mb:.1f} MB)")

print(f"\n  Total: {len(parquet_files)} files")

if len(parquet_files) < 5:
    print(f"\n  ⚠️ Warning: Expected 5 files, found {len(parquet_files)}")
    print("  Upload the missing files before continuing.")

In [0]:
# === Read all Parquet files and combine them into a single Bronze DataFrame ===

print("Reading and combining all DistriNet files...")
print("=" * 60)

combined_dfs = []

for parquet_file in parquet_files:
    full_path = os.path.join(volume_path, parquet_file)
    
    # Extract day name from filename (e.g., "Benign-Monday.parquet" -> "Monday")
    file_stem = Path(parquet_file).stem  # "Benign-Monday"
    day_name = file_stem.split('-')[-1]  # "Monday"
    
    print(f"\nIngesting {parquet_file} (day: {day_name})...")
    
    # Read the Parquet file (with column name cleaning + metadata)
    df = ingest_parquet_to_bronze_memory(spark, full_path)
    
    # Add a 'day_of_week' column to track the source
    df = df.withColumn("day_of_week", F.lit(day_name))
    
    row_count = df.count()
    col_count = len(df.columns)
    print(f"  → {row_count} rows, {col_count} columns")
    
    combined_dfs.append((day_name, df))

print("\n" + "=" * 60)
print(f"✓ All files read: {len(combined_dfs)} DataFrames")

In [0]:
# === Union all DataFrames into one combined Bronze DataFrame ===
print("Combining all DataFrames...")

# Use unionByName to safely union even if column order differs
df_combined = combined_dfs[0][1]  # Start with the first DataFrame

for day_name, df in combined_dfs[1:]:
    print(f"  Union with {day_name}...")
    df_combined = df_combined.unionByName(df)

total_rows = df_combined.count()
total_cols = len(df_combined.columns)

print(f"\n✓ Combined Bronze DataFrame ready")
print(f"  Total rows: {total_rows}")
print(f"  Total columns: {total_cols}")

# Show distribution by day
print("\n--- Row count by day_of_week ---")
df_combined.groupBy("day_of_week").count().orderBy("day_of_week").show()

In [0]:
# === Overwrite the Bronze table with combined data ===
print("Overwriting Bronze Delta table with combined data...")

(df_combined.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cybersecurity.bronze"))

print("✓ Bronze table updated")

# Verify
print("\n--- Bronze table summary ---")
spark.sql("""
    SELECT day_of_week, COUNT(*) AS row_count
    FROM workspace.cybersecurity.bronze
    GROUP BY day_of_week
    ORDER BY day_of_week
""").show()

total = spark.sql("SELECT COUNT(*) FROM workspace.cybersecurity.bronze").collect()[0][0]
print(f"Total rows in Bronze: {total}")

In [0]:
# === Read Bronze from Delta and run Silver cleaning ===
print("Reading Bronze from Delta table...")
df_bronze = spark.read.table("workspace.cybersecurity.bronze")
print(f"  ✓ Bronze loaded: {df_bronze.count()} rows")

print("\nRunning Silver cleaning pipeline on combined data...")
print("=" * 60)
df_silver = clean_bronze_to_silver(df_bronze)
print("=" * 60)

print(f"\n✓ Silver DataFrame: {df_silver.count()} rows, {len(df_silver.columns)} columns")

In [0]:
# === Overwrite Silver Delta table ===
print("Overwriting Silver Delta table...")

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cybersecurity.silver"))

print("✓ Silver table updated")

# Show label distribution (now should have attacks!)
print("\n--- Label distribution (Silver) ---")
spark.sql("""
    SELECT label, COUNT(*) AS count
    FROM workspace.cybersecurity.silver
    GROUP BY label
    ORDER BY count DESC
""").show(20, truncate=False)

In [0]:
# === Read Silver from Delta and run Gold transformation ===
print("Reading Silver from Delta table...")
df_silver = spark.read.table("workspace.cybersecurity.silver")
print(f"  ✓ Silver loaded: {df_silver.count()} rows")

print("\nRunning Gold transformation pipeline...")
print("=" * 60)
df_gold = transform_silver_to_gold(df_silver)
print("=" * 60)

print(f"\n✓ Gold DataFrame: {df_gold.count()} rows, {len(df_gold.columns)} columns")

# Write to Gold Delta table
print("\nOverwriting Gold Delta table...")
(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cybersecurity.gold"))

print("✓ Gold table updated")

In [0]:
# === Now the dataset has BOTH Benign AND Attacks! ===
print("=" * 60)
print("Combined Dataset Analysis")
print("=" * 60)

print("\n--- is_attack distribution (Gold) ---")
spark.sql("""
    SELECT is_attack, COUNT(*) AS count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.cybersecurity.gold), 2) AS percentage
    FROM workspace.cybersecurity.gold
    GROUP BY is_attack
    ORDER BY is_attack
""").show()

print("\n--- Label distribution (Gold) ---")
spark.sql("""
    SELECT label, COUNT(*) AS count
    FROM workspace.cybersecurity.gold
    GROUP BY label
    ORDER BY count DESC
""").show(20, truncate=False)

print("\n--- Attack category distribution (Gold) ---")
if "attack_category" in spark.read.table("workspace.cybersecurity.gold").columns:
    spark.sql("""
        SELECT attack_category, COUNT(*) AS count
        FROM workspace.cybersecurity.gold
        GROUP BY attack_category
        ORDER BY count DESC
    """).show()

print("\n--- Distribution by day_of_week (Gold) ---")
spark.sql("""
    SELECT day_of_week, 
           COUNT(*) AS total_flows,
           SUM(is_attack) AS attack_count,
           ROUND(SUM(is_attack) * 100.0 / COUNT(*), 2) AS attack_percentage
    FROM workspace.cybersecurity.gold
    GROUP BY day_of_week
    ORDER BY day_of_week
""").show()

In [0]:
# === Delta MERGE demo: incremental update ===
print("=" * 60)
print("Delta MERGE Demo - Incremental Update")
print("=" * 60)

print("""
Delta MERGE is the equivalent of SQL UPSERT (UPDATE + INSERT).
It's used for:
- Incremental ETL (add new data without rewriting the whole table)
- SCD Type 2 (slowly changing dimensions)
- Idempotent updates (re-running the same load won't duplicate data)
""")

# Demo: count Gold versions before MERGE
print("--- Gold table history BEFORE MERGE ---")
spark.sql("DESCRIBE HISTORY workspace.cybersecurity.gold").select(
    "version", "operation", "timestamp"
).show(truncate=False)

# Create a small "delta" DataFrame (simulate new incoming data)
print("\n--- Creating a small delta DataFrame (10 random rows from Bronze) ---")
delta_df = spark.sql("""
    SELECT * FROM workspace.cybersecurity.bronze
    ORDER BY rand()
    LIMIT 10
""")
print(f"  Delta rows: {delta_df.count()}")

# Apply the same transformations to delta_df
delta_silver = clean_bronze_to_silver(delta_df)
delta_gold = transform_silver_to_gold(delta_silver)

print(f"  Delta Gold rows: {delta_gold.count()}")

# MERGE into the Gold table (using a synthetic key for demo)
# Note: For real MERGE, you'd need a unique key. DistriNet doesn't have one
# so we use a simplified append-based approach for demo purposes.
print("\n--- MERGE pattern example (using append for demo) ---")
print("  In production, MERGE would look like:")
print("""
    MERGE INTO workspace.cybersecurity.gold AS target
    USING delta_gold AS source
    ON target.<unique_key> = source.<unique_key>
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# For demo, just append the new rows
print("\n--- Appending 10 new rows to Gold table ---")
(delta_gold.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.cybersecurity.gold"))

# Verify
print("\n--- Gold row count after append ---")
new_count = spark.sql("SELECT COUNT(*) FROM workspace.cybersecurity.gold").collect()[0][0]
print(f"  New total: {new_count} rows (was 350718*X, now {new_count})")

print("\n--- Gold table history AFTER MERGE/append ---")
spark.sql("DESCRIBE HISTORY workspace.cybersecurity.gold").select(
    "version", "operation", "timestamp"
).show(truncate=False)

In [0]:
# === Final SQL queries on the complete dataset ===
print("=" * 60)
print("Final SQL Queries on Multi-Day DistriNet Dataset")
print("=" * 60)

print("\n--- 1. Total rows per layer ---")
spark.sql("""
    SELECT 'bronze' AS layer, COUNT(*) AS rows FROM workspace.cybersecurity.bronze
    UNION ALL
    SELECT 'silver' AS layer, COUNT(*) AS rows FROM workspace.cybersecurity.silver
    UNION ALL
    SELECT 'gold' AS layer, COUNT(*) AS rows FROM workspace.cybersecurity.gold
""").show()

print("\n--- 2. Attack rate per day ---")
spark.sql("""
    SELECT day_of_week,
           COUNT(*) AS total_flows,
           SUM(is_attack) AS attacks,
           ROUND(SUM(is_attack) * 100.0 / COUNT(*), 2) AS attack_rate
    FROM workspace.cybersecurity.gold
    GROUP BY day_of_week
    ORDER BY day_of_week
""").show()

print("\n--- 3. Top attack types ---")
spark.sql("""
    SELECT label, COUNT(*) AS count
    FROM workspace.cybersecurity.gold
    WHERE is_attack = 1
    GROUP BY label
    ORDER BY count DESC
""").show(20, truncate=False)

print("\n--- 4. Average packet count by attack type ---")
spark.sql("""
    SELECT label,
           COUNT(*) AS flow_count,
           ROUND(AVG(total_packets), 2) AS avg_packets,
           ROUND(AVG(total_bytes), 2) AS avg_bytes
    FROM workspace.cybersecurity.gold
    GROUP BY label
    ORDER BY flow_count DESC
""").show(20, truncate=False)

print("\n--- 5. SYN flag distribution (DoS detection signal) ---")
spark.sql("""
    SELECT has_syn_flag,
           COUNT(*) AS count,
           SUM(is_attack) AS attacks,
           ROUND(SUM(is_attack) * 100.0 / COUNT(*), 2) AS attack_rate
    FROM workspace.cybersecurity.gold
    GROUP BY has_syn_flag
""").show()